# Chấm điểm động tác chống đẩy — Bước 5: ST-GCN

Đầu vào: `pushup_dataset.npz` của bước 3 (cùng dữ liệu, cùng cách chia fold với LSTM ở bước 4 nên so sánh trực tiếp được).

Notebook này train hai phiên bản ST-GCN:
1. **Chỉ nhãn good/bad** — để so với LSTM.
2. **Đa nhiệm** — ngoài good/bad, mô hình còn phải đoán loại lỗi (`nong`, `hong`, `sap`, `tay`) theo bảng nhãn tự gán ở mục 2.

Nên bật GPU (Settings → Accelerator) cho nhanh.

## 1. Đọc dữ liệu

In [ ]:
import os, glob, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

found = glob.glob("/kaggle/working/**/pushup_dataset.npz", recursive=True) \
      + glob.glob("/kaggle/input/**/pushup_dataset.npz", recursive=True)
assert found, "Không thấy pushup_dataset.npz. Hãy Add Input dataset của bước 3."
OUT_DIR = "/kaggle/working/stgcn"
os.makedirs(OUT_DIR, exist_ok=True)

data = np.load(found[0])
X = data["X2d"]                      # (N, T, 12, 3): x, y, visibility
y, view, subject, fold, video = data["y"], data["view"], data["subject"], data["fold"], data["video"]
EDGES = [tuple(e) for e in data["edges"]]
VIEW_NAMES = ["front", "side", "diagonal"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("X:", X.shape, "| thiết bị:", device)

## 2. Nhãn loại lỗi (tự gán bằng cách xem 24 video bad)

Dataset chỉ ghi good/bad cho cả video. Bảng dưới là kết quả xem từng video bad:
- `nong`: xuống không đủ sâu
- `hong`: hông võng, mông nhô, hoặc gối chạm sàn
- `sap`: đổ sập người xuống sàn
- `tay`: tay đặt sai vị trí hoặc khuỷu bẻ ngang

`ALL_BAD[s] = False` nghĩa là trong video bad của người `s` có lẫn rep đúng, nên nhãn "sai" của các rep đó **không chắc chắn**.
Các rep này vẫn được dùng, nhưng kết quả được báo cáo thêm trên **phần nhãn sạch** (bỏ các rep không chắc chắn).

In [ ]:
ERR_NAMES = ["nong", "hong", "sap", "tay"]
ERR = {
    1: ["tay", "nong"], 2: ["hong"], 3: ["hong"], 4: ["tay", "nong"], 5: ["nong", "hong"], 6: ["hong"],
    7: ["sap"], 8: ["hong", "nong"], 9: ["nong", "tay", "hong"], 10: ["hong", "tay"], 11: ["tay"], 12: ["sap"],
    13: ["tay"], 14: ["tay"], 15: ["hong", "sap"], 16: ["nong"], 17: ["sap"], 18: ["hong"],
    19: ["hong", "sap"], 20: ["nong"], 21: ["nong"], 22: ["nong"], 23: ["hong"], 24: ["nong"],
}
NOT_ALL_BAD = [1, 3, 11, 12, 13, 15, 17]       # video bad có lẫn rep đúng

E = np.zeros((len(y), len(ERR_NAMES)), dtype=np.float32)      # nhãn loại lỗi của từng rep
for i in np.where(y == 1)[0]:
    for e in ERR[int(subject[i])]:
        E[i, ERR_NAMES.index(e)] = 1
clean = (y == 0) | ~np.isin(subject, NOT_ALL_BAD)             # rep có nhãn chắc chắn

print("Số rep theo loại lỗi:", dict(zip(ERR_NAMES, E.sum(0).astype(int))))
print("Tỉ lệ rep có nhãn chắc chắn:", round(float(clean.mean()), 3))

## 3. Tăng cường dữ liệu

Giống hệt bước 4: lật trái/phải, co giãn và xoay nhẹ, che ngẫu nhiên khớp.

In [ ]:
SWAP_LR = [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10]


def augment(x, rng):
    x = x.copy()
    if rng.random() < 0.5:
        x = x[:, SWAP_LR]
        x[..., 0] = -x[..., 0]
    scale = rng.uniform(0.9, 1.1)
    a = np.deg2rad(rng.uniform(-10, 10))
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]], dtype=np.float32)
    x[..., :2] = (x[..., :2] @ rot.T) * scale
    drop = rng.random(x.shape[1]) < 0.1
    x[:, drop] = 0.0
    return x


class RepDataset(torch.utils.data.Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = idx, train
        self.rng = np.random.default_rng(0)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = augment(X[j], self.rng) if self.train else X[j]
        target = np.concatenate([[y[j]], E[j]]).astype(np.float32)     # [sai?, nong, hong, sap, tay]
        return torch.from_numpy(x), int(view[j]), torch.from_numpy(target)

## 4. Mô hình ST-GCN

Khung xương là một **đồ thị**: 12 khớp là đỉnh, 12 xương là cạnh (ma trận kề `A`, có thêm cạnh tự nối, chuẩn hoá đối xứng).

Mỗi khối ST-GCN gồm:
1. **Tích chập đồ thị theo không gian**: mỗi khớp tổng hợp đặc trưng của các khớp nối với nó (nhân với `A`).
   `edge_importance` là trọng số học được cho từng cạnh.
2. **Tích chập theo thời gian**: kernel 9 khung trên từng khớp.
3. Kết nối tắt (residual).

4 khối (32 → 32 → 64 → 64 kênh, khối thứ ba giảm một nửa độ dài thời gian) → lấy trung bình trên mọi khớp và mọi khung
→ ghép one-hot góc quay → lớp tuyến tính ra 5 số: `[sai?, nong, hong, sap, tay]`.

Mô hình được giữ nhỏ (khoảng 100 nghìn tham số) vì chỉ có 24 người tập.

In [ ]:
def normalized_adjacency(n, edges):
    A = np.eye(n, dtype=np.float32)
    for a, b in edges:
        A[a, b] = A[b, a] = 1
    d = A.sum(1)
    return torch.tensor(A / np.sqrt(d[:, None] * d[None, :]))


class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, A, stride=1, kernel_t=9, dropout=0.3):
        super().__init__()
        self.register_buffer("A", A)
        self.edge_importance = nn.Parameter(torch.ones_like(A))
        self.gcn = nn.Conv2d(c_in, c_out, 1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, (kernel_t, 1), (stride, 1), (kernel_t // 2, 0)),
            nn.BatchNorm2d(c_out), nn.Dropout(dropout))
        if c_in == c_out and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride, 1)), nn.BatchNorm2d(c_out))

    def forward(self, x):                                   # x: (B, C, T, V)
        h = torch.einsum("bctv,vw->bctw", self.gcn(x), self.A * self.edge_importance)
        return torch.relu(self.tcn(h) + self.residual(x))


class STGCN(nn.Module):
    def __init__(self, n_out=5, c_in=3, channels=(32, 32, 64, 64), n_joints=12, n_views=3, dropout=0.3):
        super().__init__()
        A = normalized_adjacency(n_joints, EDGES)
        self.n_views = n_views
        self.input_bn = nn.BatchNorm1d(c_in * n_joints)
        blocks, c = [], c_in
        for i, c_out in enumerate(channels):
            blocks.append(STGCNBlock(c, c_out, A, stride=2 if i == 2 else 1, dropout=dropout))
            c = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(c + n_views, n_out))

    def forward(self, x, view_id):                          # x: (B, T, V, C)
        b, t, v, c = x.shape
        h = self.input_bn(x.permute(0, 2, 3, 1).reshape(b, v * c, t))
        h = h.reshape(b, v, c, t).permute(0, 2, 3, 1)       # (B, C, T, V)
        h = self.blocks(h).mean(dim=(2, 3))
        return self.head(torch.cat([h, nn.functional.one_hot(view_id, self.n_views).float()], dim=-1))


print("Số tham số:", sum(p.numel() for p in STGCN().parameters()))

## 5. Hàm train

Giống bước 4 (AdamW, dừng sớm theo loss validation của đầu ra "sai?"), chỉ khác hàm mất mát:

`loss = BCE(sai?) + err_weight × BCE(4 loại lỗi)`

- `err_weight = 0`: chỉ học good/bad.
- `err_weight = 1`: học đa nhiệm.

In [ ]:
@torch.no_grad()
def predict(model, idx):
    model.eval()
    loader = torch.utils.data.DataLoader(RepDataset(idx, train=False), batch_size=256)
    return torch.cat([torch.sigmoid(model(xb.to(device), vb.to(device))).cpu() for xb, vb, _ in loader]).numpy()


def train_one_fold(k, err_weight, epochs=80, patience=12, lr=1e-3, seed=0):
    random.seed(seed + k); np.random.seed(seed + k); torch.manual_seed(seed + k)
    test_idx = np.where(fold == k)[0]
    val_idx = np.where(fold == (k + 1) % 5)[0]
    train_idx = np.where((fold != k) & (fold != (k + 1) % 5))[0]

    model = STGCN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    weight = torch.tensor([1.0] + [err_weight] * len(ERR_NAMES), device=device)
    loader = torch.utils.data.DataLoader(RepDataset(train_idx, train=True), batch_size=64, shuffle=True)
    y_val = torch.tensor(y[val_idx], dtype=torch.float32)

    best_loss, best_state, bad_epochs, history = np.inf, None, 0, []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, vb, tb in loader:
            opt.zero_grad()
            logits = model(xb.to(device), vb.to(device))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, tb.to(device), reduction="none") * weight).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(tb)
        val_loss = nn.functional.binary_cross_entropy(torch.tensor(predict(model, val_idx)[:, 0]), y_val).item()
        history.append((total / len(train_idx), val_loss))
        if val_loss < best_loss:
            best_loss, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, test_idx, predict(model, test_idx), history


def run_cv(err_weight, name):
    prob = np.zeros((len(y), 1 + len(ERR_NAMES)), dtype=np.float32)
    for k in range(5):
        model, test_idx, p, history = train_one_fold(k, err_weight)
        prob[test_idx] = p
        acc = ((p[:, 0] > 0.5) == y[test_idx]).mean()
        print(f"[{name}] fold {k}: {len(history)} epoch, accuracy test = {acc:.3f}")
        torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{name}_fold{k}.pt"))
    return prob

## 6. Train hai phiên bản

In [ ]:
prob_binary = run_cv(err_weight=0.0, name="stgcn_good_bad")
prob_multi = run_cv(err_weight=1.0, name="stgcn_da_nhiem")
np.savez(os.path.join(OUT_DIR, "stgcn_predictions.npz"), prob_binary=prob_binary, prob_multi=prob_multi,
         y=y, E=E, view=view, subject=subject, fold=fold, clean=clean)

## 7. Kết quả phân loại đúng/sai theo từng rep

Lớp dương = rep **sai**. `false_alarm` = tỉ lệ rep đúng bị báo là sai.
So sánh trực tiếp với bảng LSTM ở bước 4 (cùng dữ liệu, cùng fold).

In [ ]:
def metrics(p_bad, mask, threshold=0.5):
    t, p = y[mask], (p_bad[mask] > threshold).astype(int)
    tp, fp = int(((p == 1) & (t == 1)).sum()), int(((p == 1) & (t == 0)).sum())
    fn, tn = int(((p == 0) & (t == 1)).sum()), int(((p == 0) & (t == 0)).sum())
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(n=int(mask.sum()), accuracy=(tp + tn) / max(len(t), 1), precision=precision, recall=recall,
                f1=2 * precision * recall / max(precision + recall, 1e-9), false_alarm=fp / max(fp + tn, 1))


rows = {}
for name, prob in [("ST-GCN good/bad", prob_binary), ("ST-GCN đa nhiệm", prob_multi)]:
    rows[f"{name} / tất cả"] = metrics(prob[:, 0], np.ones(len(y), bool))
    for v, view_name in enumerate(VIEW_NAMES):
        rows[f"{name} / {view_name}"] = metrics(prob[:, 0], view == v)
    rows[f"{name} / chỉ nhãn sạch"] = metrics(prob[:, 0], clean)
result = pd.DataFrame(rows).T
result.to_csv(os.path.join(OUT_DIR, "stgcn_results.csv"))
display(result.round(3))

## 8. Kết quả theo video

Nhãn gốc của dataset là theo **video**, nên đây là cách đánh giá khớp với nhãn nhất:
lấy trung bình xác suất "sai" của mọi rep trong video, lớn hơn 0.5 thì coi video đó là bad.

In [ ]:
rows = {}
for name, prob in [("ST-GCN good/bad", prob_binary), ("ST-GCN đa nhiệm", prob_multi)]:
    per_video = pd.DataFrame({"video": video, "p": prob[:, 0], "y": y, "view": np.array(VIEW_NAMES)[view]}) \
        .groupby("video").agg(p=("p", "mean"), y=("y", "first"), view=("view", "first"))
    correct = (per_video.p > 0.5) == per_video.y
    rows[name] = {"số video": len(per_video), "tất cả": correct.mean(), **correct.groupby(per_video["view"]).mean().to_dict()}
display(pd.DataFrame(rows).T.round(3))

## 9. Góc quay nào thấy được lỗi nào?

AUC của từng đầu ra loại lỗi (phiên bản đa nhiệm), tách theo góc quay.
AUC = 0.5 là đoán bừa, 1.0 là hoàn hảo. Đây là bảng trả lời câu hỏi "góc nào phát hiện được lỗi nào".

In [ ]:
from sklearn.metrics import roc_auc_score

table = {}
for i, e in enumerate(ERR_NAMES):
    table[e] = {view_name: roc_auc_score(E[view == v, i], prob_multi[view == v, 1 + i]) for v, view_name in enumerate(VIEW_NAMES)}
    table[e]["tất cả"] = roc_auc_score(E[:, i], prob_multi[:, 1 + i])
display(pd.DataFrame(table).T.round(2))

## 10. Kết quả theo từng người

Hàng `0`: tỉ lệ rep **đúng** được nhận là đúng. Hàng `1`: tỉ lệ rep **sai** được nhận là sai.
Người nào có hàng `1` gần 0 nghĩa là mô hình không nhận ra kiểu sai của người đó.

In [ ]:
per_subject = pd.DataFrame({"subject": subject, "nhãn": y, "dung": (prob_binary[:, 0] > 0.5) == y})
display(per_subject.pivot_table(index="nhãn", columns="subject", values="dung", aggfunc="mean").round(2))

## 11. Kết quả

Tải thư mục `stgcn` hoặc Save Version để giữ lại.

In [ ]:
!ls -lh "{OUT_DIR}"